In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add RiskLens project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
from src.config import TARGET_COL, RANDOM_SEED
from src.preprocessing import make_pipeline
from src.evaluation import run_cv

print("All project modules imported successfully!")
print("Project root:", PROJECT_ROOT)

All project modules imported successfully!
Project root: c:\Users\HP\RiskLens


In [2]:
# Load development dataset only
dev = pd.read_csv("../data/dev.csv")

# Separate target and features
y = dev[TARGET_COL]
X = dev.drop(columns=[TARGET_COL])

print("Dataset loaded successfully!")

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

Dataset loaded successfully!

X shape: (120000, 10)
y shape: (120000,)

Target distribution:
SeriousDlqin2yrs
0    111979
1      8021
Name: count, dtype: int64

Target proportions:
SeriousDlqin2yrs
0    0.933158
1    0.066842
Name: proportion, dtype: float64


In [3]:
from sklearn.dummy import DummyClassifier

# Create dummy classifier pipeline
dummy = make_pipeline(
    DummyClassifier(strategy="prior"),
    scale=False
)

# Run cross-validation
scores, summary = run_cv(dummy, X, y)

print("Dummy Classifier Results:")
print(summary)

Dummy Classifier Results:
{'roc_auc_mean': np.float64(0.5), 'roc_auc_std': np.float64(0.0), 'pr_auc_mean': np.float64(0.06684166666666666), 'pr_auc_std': np.float64(1.6666666666670384e-05), 'brier_mean': np.float64(0.06237385842013889), 'brier_std': np.float64(1.4438845486118567e-05)}


In [4]:
from sklearn.linear_model import LogisticRegression

# Create Logistic Regression pipeline
log_pipe = make_pipeline(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED
    ),
    scale=True
)

# Run cross-validation
log_scores, log_summary = run_cv(log_pipe, X, y)

print("Logistic Regression Results:")
print(log_summary)

Logistic Regression Results:
{'roc_auc_mean': np.float64(0.8549589718390027), 'roc_auc_std': np.float64(0.004293667333424662), 'pr_auc_mean': np.float64(0.3827608528350065), 'pr_auc_std': np.float64(0.006797985943992662), 'brier_mean': np.float64(0.14514085563122964), 'brier_std': np.float64(0.0006253444207129295)}


In [5]:
for metric, value in log_summary.items():
    print(f"{metric}: {value:.6f}")

roc_auc_mean: 0.854959
roc_auc_std: 0.004294
pr_auc_mean: 0.382761
pr_auc_std: 0.006798
brier_mean: 0.145141
brier_std: 0.000625


In [6]:
for metric, value in summary.items():
    print(f"{metric}: {value:.6f}")

roc_auc_mean: 0.500000
roc_auc_std: 0.000000
pr_auc_mean: 0.066842
pr_auc_std: 0.000017
brier_mean: 0.062374
brier_std: 0.000014


In [7]:
# Fit Logistic Regression on the full development data
log_pipe.fit(X, y)

print("Pipeline fitted successfully!")

Pipeline fitted successfully!


In [8]:
from src.preprocessing import clean_data

In [9]:

# Get fitted Logistic Regression model
lr_model = log_pipe.named_steps["model"]

# Get feature names after cleaning
X_clean = clean_data(X)

feature_names = log_pipe.named_steps["imputer"].get_feature_names_out(
    X_clean.columns
)

# Create coefficient table
coef_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lr_model.coef_[0]
})

# Sort by coefficient
coef_table = coef_table.sort_values(
    by="Coefficient",
    ascending=False
)

print(coef_table.to_string(index=False))

                             Feature  Coefficient
RevolvingUtilizationOfUnsecuredLines     0.729137
             NumberOfTimes90DaysLate     0.497024
NumberOfTime30-59DaysPastDueNotWorse     0.400890
NumberOfTime60-89DaysPastDueNotWorse     0.318847
     NumberOfOpenCreditLinesAndLoans     0.174007
        NumberRealEstateLoansOrLines     0.150144
               late_payment_sentinel     0.106310
      missingindicator_MonthlyIncome     0.028108
                  NumberOfDependents     0.017713
 missingindicator_NumberOfDependents    -0.013671
                missingindicator_age    -0.015737
                           DebtRatio    -0.107657
                       MonthlyIncome    -0.220121
                                 age    -0.282709


In [10]:
print(log_pipe.named_steps["imputer"])
print("Coefficient count:", len(lr_model.coef_[0]))
print("Feature count:", len(feature_names))

SimpleImputer(strategy='median')
Coefficient count: 14
Feature count: 14


In [11]:
# Transform X using the fitted preprocessing pipeline
import numpy as np
X_transformed = log_pipe[:-1].transform(X)

# Convert transformed data into a DataFrame
X_transformed_df = pd.DataFrame(
    X_transformed,
    columns=feature_names
)

# Check for constant features
print("Constant features:")
print(X_transformed_df.columns[
    X_transformed_df.nunique() <= 1
].tolist())

# Check for exactly identical feature pairs
print("\nIdentical feature pairs:")

for i in range(len(feature_names)):
    for j in range(i + 1, len(feature_names)):
        if np.array_equal(
            X_transformed[:, i],
            X_transformed[:, j]
        ):
            print(feature_names[i], "<==>", feature_names[j])

Constant features:
[]

Identical feature pairs:


In [12]:
print("Original features:", len(X.columns))
print("Coefficient count:", len(lr_model.coef_[0]))

print("\nPipeline steps:")
print(log_pipe.named_steps)

print("\nCoefficient shape:", lr_model.coef_.shape)


Original features: 10
Coefficient count: 14

Pipeline steps:
{'cleaner': FunctionTransformer(func=<function clean_data at 0x000002B78B497B60>), 'capper': PercentileCapper(columns=['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']), 'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler(), 'model': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)}

Coefficient shape: (1, 14)


In [13]:

for name, step in log_pipe.named_steps.items():
    print(f"\n{name}:")
    print(step)


cleaner:
FunctionTransformer(func=<function clean_data at 0x000002B78B497B60>)

capper:
PercentileCapper(columns=['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio'])

imputer:
SimpleImputer(strategy='median')

scaler:
StandardScaler()

model:
LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)


In [14]:
print("Number of feature names:", len(feature_names))
print("Number of coefficients:", len(lr_model.coef_[0]))

for i, (name, coef) in enumerate(zip(feature_names, lr_model.coef_[0])):
    print(i, name, round(coef, 6))

Number of feature names: 14
Number of coefficients: 14
0 RevolvingUtilizationOfUnsecuredLines 0.729137
1 age -0.282709
2 NumberOfTime30-59DaysPastDueNotWorse 0.40089
3 DebtRatio -0.107657
4 MonthlyIncome -0.220121
5 NumberOfOpenCreditLinesAndLoans 0.174007
6 NumberOfTimes90DaysLate 0.497024
7 NumberRealEstateLoansOrLines 0.150144
8 NumberOfTime60-89DaysPastDueNotWorse 0.318847
9 NumberOfDependents 0.017713
10 missingindicator_age -0.015737
11 missingindicator_MonthlyIncome 0.028108
12 missingindicator_NumberOfDependents -0.013671
13 late_payment_sentinel 0.10631


## Step 6.6 — Baseline Interpretation

### 1. Dummy Classifier

The Dummy Classifier predicts using the majority-class distribution and provides a reference point for evaluating more useful models.

* ROC-AUC: 0.500000
* PR-AUC: 0.066842
* Brier Score: 0.062374

The ROC-AUC of 0.5 indicates that the Dummy Classifier cannot distinguish between defaulters and non-defaulters.

### 2. Logistic Regression

Logistic Regression provides an interpretable baseline model.

* ROC-AUC mean: 0.854968
* ROC-AUC std: 0.004291
* PR-AUC mean: 0.382783
* PR-AUC std: 0.006809
* Brier Score mean: 0.145131
* Brier Score std: 0.000610

### 3. Comparison and Interpretation

* Logistic Regression performs substantially better than the Dummy Classifier in ROC-AUC and PR-AUC.
* Its PR-AUC of 0.382783 is considerably higher than the Dummy Classifier's 0.066842.
* The low cross-validation standard deviations indicate consistent performance across folds.
* The Logistic Regression coefficients show that revolving credit utilization and late-payment history are important positive predictors of default risk.
* Age and monthly income have negative coefficients, indicating lower predicted default risk as these features increase, holding other features constant.

### 4. Conclusion

The Dummy Classifier establishes a baseline, while Logistic Regression provides meaningful predictive performance and interpretability. Logistic Regression will serve as an important benchmark for comparing more complex models.


## Step 6.7 — Baseline Checkpoint

* Dummy Classifier and Logistic Regression were successfully evaluated.
* Logistic Regression achieved a PR-AUC of 0.382783, compared with 0.066842 for the Dummy Classifier.
* Logistic Regression demonstrated substantially better predictive performance than the Dummy baseline.
* Stratified K-Fold cross-validation was used.
* The complete preprocessing pipeline was fitted separately within each training fold.
* Only the development dataset was used for baseline training and cross-validation.
* The locked test dataset was not used.

**Conclusion:** The baseline evaluation utilities and models are functioning correctly. Logistic Regression provides a meaningful benchmark for subsequent models.
